In [10]:
import os
import torch
import cv2
import numpy as np
from PIL import Image
from transformers import (
    DetrImageProcessor,
    DetrForObjectDetection,
    ViTForImageClassification,
    ViTImageProcessor
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [11]:
print("Loading object detection model...")
det_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
det_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device)

Loading object detection model...


Loading weights: 100%|██████████| 530/530 [00:00<00:00, 2947.85it/s]
DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
print("Loading Pokémon classifier...")
classifier_id = "skshmjn/Pokemon-classifier-gen9-1025"
clf_model = ViTForImageClassification.from_pretrained(classifier_id).to(device)
clf_processor = ViTImageProcessor.from_pretrained(classifier_id)

Loading Pokémon classifier...


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 3538.71it/s]


In [13]:
def detect_objects(image_pil, threshold=0.8):
    inputs = det_processor(images=image_pil, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = det_model(**inputs)

    target_sizes = torch.tensor([image_pil.size[::-1]]).to(device)
    results = det_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=threshold
    )[0]

    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results["labels"].cpu().numpy()

    return boxes, scores, labels

In [14]:
def crop_and_save(image_pil, boxes, save_dir="crops"):
    os.makedirs(save_dir, exist_ok=True)
    crops = []

    img_np = np.array(image_pil)

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)

        crop = img_np[y1:y2, x1:x2]
        crop_pil = Image.fromarray(crop)

        save_path = f"{save_dir}/pokemon_{i}.png"
        crop_pil.save(save_path)

        crops.append((i, crop_pil, save_path))

    return crops

In [15]:
def classify_pokemon(image_pil):
    inputs = clf_processor(images=image_pil, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = clf_model(**inputs)

    pred_id = outputs.logits.argmax(-1).item()
    name = clf_model.config.id2label[pred_id]

    return pred_id + 1, name

In [16]:
def run_pipeline(image_path):
    print("\nProcessing:", image_path)

    image = Image.open(image_path).convert("RGB")

    # 1️⃣ Detect objects
    boxes, scores, labels = detect_objects(image)

    print(f"Found {len(boxes)} possible Pokémon")

    if len(boxes) == 0:
        print("No objects detected.")
        return

    # 2️⃣ Crop them
    crops = crop_and_save(image, boxes)

    # 3️⃣ Classify each crop
    print("\n===== RESULTS =====")
    for i, crop_img, path in crops:
        dex, name = classify_pokemon(crop_img)
        print(f"Crop {i}: {name} (Pokédex #{dex}) → saved at {path}")

In [17]:
if __name__ == "__main__":
    run_pipeline("img3.jpg")


Processing: img3.jpg
Found 12 possible Pokémon

===== RESULTS =====
Crop 0: Torterra (Pokédex #389) → saved at crops/pokemon_0.png
Crop 1: Feraligatr (Pokédex #160) → saved at crops/pokemon_1.png
Crop 2: Typhlosion (Pokédex #157) → saved at crops/pokemon_2.png
Crop 3: Monferno (Pokédex #391) → saved at crops/pokemon_3.png
Crop 4: Wartortle (Pokédex #8) → saved at crops/pokemon_4.png
Crop 5: Prinplup (Pokédex #394) → saved at crops/pokemon_5.png
Crop 6: Swampert (Pokédex #260) → saved at crops/pokemon_6.png
Crop 7: Charizard (Pokédex #6) → saved at crops/pokemon_7.png
Crop 8: Empoleon (Pokédex #395) → saved at crops/pokemon_8.png
Crop 9: Pignite (Pokédex #499) → saved at crops/pokemon_9.png
Crop 10: Delphox (Pokédex #655) → saved at crops/pokemon_10.png
Crop 11: Blastoise (Pokédex #9) → saved at crops/pokemon_11.png
